# TennisMyLife — Gallica 1903 Colab worker — A100 AUTO-TUNED + LIVE LOG
ALTO is strictly rate-limited to one Gallica request every 16 seconds. RapidOCR never requests Gallica. The final cell streams child-process logs live with [TUNE], [ALTO], [RAPID] and a [RUNNER] heartbeat every 5 seconds.


In [ ]:
!rm -rf /content/Tennis-OCR-Pipeline
!git clone -q https://github.com/Tennismylife/Tennis-OCR-Pipeline.git /content/Tennis-OCR-Pipeline
!pip -q uninstall -y onnxruntime onnxruntime-gpu >/dev/null 2>&1 || true
!pip -q install -r /content/Tennis-OCR-Pipeline/colab/requirements.txt


In [ ]:
import subprocess, onnxruntime as ort, os
subprocess.run(['nvidia-smi','--query-gpu=name,memory.total,driver_version','--format=csv,noheader'],check=True)
providers=ort.get_available_providers(); print('ONNX Runtime providers:',providers)
if 'CUDAExecutionProvider' not in providers: raise RuntimeError('CUDAExecutionProvider not available')
print('GPU CHECK OK — A100 CUDA active; CPU cores:', os.cpu_count())


In [ ]:
from google.colab import files
import base64
uploaded=files.upload()
if not uploaded: raise RuntimeError('No SSH key uploaded')
key_name,key_bytes=next(iter(uploaded.items()))
if key_name.endswith('.pub'): raise RuntimeError('Upload tml_colab_ed25519, not .pub')
KEY_B64=base64.b64encode(key_bytes).decode(); print('SSH key loaded:',key_name)


In [ ]:
import sys
sys.path.insert(0,'/content/Tennis-OCR-Pipeline/colab')
from worker import connect_sftp
VPS_HOST='vibrant-lovelace.82-165-11-122.plesk.page'; VPS_USER='andre'; VPS_PORT=2222
VPS_MANIFEST='/home/andre/GallicaJobs/gallica-1903-all-tennis/GALlica_1903_ALL_TENNIS/00_MANIFEST/colab_active_claims.tsv'
VPS_REMOTE_CACHE='/home/andre/GallicaJobs/gallica-1903-all-tennis/GALlica_1903_ALL_TENNIS/ocr_cache_latin_full/targeted_remaining_1903'
VPS_ALTO_CACHE='/home/andre/GallicaJobs/_shared/alto_cache'
tr,sftp=connect_sftp(VPS_HOST,VPS_USER,KEY_B64,VPS_PORT); sftp.get(VPS_MANIFEST,'/content/colab_claim.tsv'); sftp.close(); tr.close()
rows=sum(1 for _ in open('/content/colab_claim.tsv',encoding='utf-8-sig'))-1; print('Claim downloaded. Rows:',rows)


In [ ]:
import csv
with open('/content/colab_claim.tsv',encoding='utf-8-sig',newline='') as f: rr=list(csv.DictReader(f,delimiter='\t'))
fields=list(rr[0]) if rr else ['ark','page','mode']
for mode,path in [('ALTO','/content/colab_alto.tsv'),('RAPID','/content/colab_rapid.tsv')]:
    subset=[r for r in rr if (r.get('mode') or '').upper()==mode]
    with open(path,'w',encoding='utf-8-sig',newline='') as f:
        w=csv.DictWriter(f,fieldnames=fields,delimiter='\t'); w.writeheader(); w.writerows(subset)
    print(mode,'rows:',len(subset))


In [ ]:
import subprocess, os, csv, json, threading, time
subprocess.run(['git','-C','/content/Tennis-OCR-Pipeline','pull','--ff-only'],check=True)
commit=subprocess.check_output(['git','-C','/content/Tennis-OCR-Pipeline','rev-parse','--short','HEAD'],text=True).strip(); print('Code commit:',commit,flush=True)
tr,sftp=connect_sftp(VPS_HOST,VPS_USER,KEY_B64,VPS_PORT); sftp.get(VPS_MANIFEST,'/content/colab_claim.tsv'); sftp.close(); tr.close()
with open('/content/colab_claim.tsv',encoding='utf-8-sig',newline='') as f: rr=list(csv.DictReader(f,delimiter='\t'))
fields=list(rr[0]) if rr else ['ark','page','mode']; counts={}
for mode,path in [('ALTO','/content/colab_alto.tsv'),('RAPID','/content/colab_rapid.tsv')]:
    subset=[r for r in rr if (r.get('mode') or '').upper()==mode]; counts[mode]=len(subset)
    with open(path,'w',encoding='utf-8-sig',newline='') as f:
        w=csv.DictWriter(f,fieldnames=fields,delimiter='\t'); w.writeheader(); w.writerows(subset)
print('LIVE CLAIM',len(rr),'ALTO',counts.get('ALTO',0),'RAPID',counts.get('RAPID',0),flush=True)
common=['--vps-host',VPS_HOST,'--vps-user',VPS_USER,'--vps-key-b64',KEY_B64,'--vps-port',str(VPS_PORT),'--remote-cache',VPS_REMOTE_CACHE]
env=dict(os.environ); env['PYTHONUNBUFFERED']='1'
def start_logged(tag,cmd):
    p=subprocess.Popen(cmd,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1,env=env)
    def pump():
        for line in iter(p.stdout.readline,''):
            if line: print(f'[{tag}] {line.rstrip()}',flush=True)
    t=threading.Thread(target=pump,daemon=True); t.start(); return p,t
def wait_with_heartbeat(label,p,others=()):
    t0=time.time()
    while p.poll() is None:
        states=' '.join(f'{name}={proc.poll() if proc.poll() is not None else "RUN"}' for name,proc in others)
        print(f'[RUNNER] phase={label} elapsed={time.time()-t0:.0f}s {states}',flush=True)
        time.sleep(5)
    return p.returncode
alto=['python','-u','/content/Tennis-OCR-Pipeline/colab/alto_safe.py','--manifest','/content/colab_alto.tsv',*common,'--remote-alto-cache',VPS_ALTO_CACHE,'--delay','16']
print('[RUNNER] starting ALTO safe branch',flush=True)
p_alto,t_alto=start_logged('ALTO',alto)
best_workers=8
if counts.get('RAPID',0)>0:
    tune=['python','-u','/content/Tennis-OCR-Pipeline/colab/rapid_autotune.py','--manifest','/content/colab_rapid.tsv','--vps-host',VPS_HOST,'--vps-user',VPS_USER,'--vps-key-b64',KEY_B64,'--vps-port',str(VPS_PORT),'--profile','HQ','--sample-pages','32','--candidates','4,8,12,16','--downloaders','8','--out','/content/tml_autotune_result.json']
    print('[RUNNER] starting A100 autotune; RapidOCR Gallica requests=0',flush=True)
    p_tune,t_tune=start_logged('TUNE',tune)
    tune_rc=wait_with_heartbeat('AUTOTUNE',p_tune,[('ALTO',p_alto)])
    t_tune.join(timeout=2)
    if tune_rc!=0:
        p_alto.terminate(); p_alto.wait(); raise RuntimeError(f'A100 autotune failed rc={tune_rc}')
    result=json.load(open('/content/tml_autotune_result.json'))
    best_workers=int(result['best_workers'])
    print('[RUNNER] AUTOTUNE SELECTED workers=',best_workers,'pages/min=',result['best_pages_per_min'],flush=True)
rapid=['python','-u','/content/Tennis-OCR-Pipeline/colab/rapid_pool.py','--manifest','/content/colab_rapid.tsv',*common,'--profile','HQ','--workers',str(best_workers),'--downloaders','8','--no-autotune']
print('[RUNNER] starting production RapidOCR workers=',best_workers,'; OCR Gallica requests=0',flush=True)
p_rapid,t_rapid=start_logged('RAPID',rapid)
rapid_rc=wait_with_heartbeat('PRODUCTION',p_rapid,[('ALTO',p_alto)])
t_rapid.join(timeout=2)
print('[RUNNER] RapidOCR finished rc=',rapid_rc,'; waiting ALTO branch',flush=True)
alto_rc=wait_with_heartbeat('ALTO_FINISH',p_alto,[])
t_alto.join(timeout=2)
print('Branches complete: ALTO=',alto_rc,'RAPID/A100=',rapid_rc,flush=True)
if alto_rc!=0 or rapid_rc!=0: raise RuntimeError(f'branch failure ALTO={alto_rc} RAPID={rapid_rc}')


The final cell streams all child-process output live. [TUNE] shows benchmark progress and GPU metrics, [RAPID] shows production OCR progress, [ALTO] shows paced Gallica ALTO work, and [RUNNER] prints a heartbeat every five seconds even if no child emits a line. RapidOCR never requests Gallica.
